# Bussiness Understanding.

# Data Understanding.

# Data Preparation

### <u>Loading the dataset<u>

In [2]:
# Importing pandas, numpy, matplotlib and seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("Salary Data - Form Responses 1.csv")
df.shape

(28223, 18)

The dataset has 28,223 rows and 18 columns.

### <u>Standardizing the column names<u>

In [4]:
df.columns

Index(['Timestamp', 'How old are you?', 'What industry do you work in?',
       'Job title',
       'If your job title needs additional context, please clarify here:',
       'What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)',
       'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.',
       'Please indicate the currency',
       'If "Other," please indicate the currency here: ',
       'If your income needs additional context, please provide it here:',
       'What country do you work in?',
       'If you're in the U.S., what state do you work in?',
       'What city do you work in?',
       'How many years of professional work experience do you have overall?',
       

In [5]:
df = df.rename(columns= {
    "How old are you?" : "age",
    "What industry do you work in?" : "industry",
    "If your job title needs additional context, please clarify here:" : "job_title_context",
    "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)" : "annual salary",
    "How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits." : "extra_compensation",
    "Please indicate the currency" : "currency",
    'If "Other," please indicate the currency here: ' : "other_currency",
    "If your income needs additional context, please provide it here:" : "income_context",
    "What country do you work in?" : "country",
    "If you're in the U.S., what state do you work in?" : "state",
    "What city do you work in?" : "city",
    "How many years of professional work experience do you have overall?" : "overall_exp",
    "How many years of professional work experience do you have in your field?" : "field_exp",
    "What is your highest level of education completed?" : "education",
    "What is your gender?" : "gender",
    "What is your race? (Choose all that apply.)" : "race"
})

df.columns = (
    df.columns.str.strip()
    .str.replace(" ", "_")
    .str.lower()
)

df.columns

Index(['timestamp', 'age', 'industry', 'job_title', 'job_title_context',
       'annual_salary', 'extra_compensation', 'currency', 'other_currency',
       'income_context', 'country', 'state', 'city', 'overall_exp',
       'field_exp', 'education', 'gender', 'race'],
      dtype='str')

### <u>Ensuring the columns contain values of the expected data type<u>

In [23]:
df.dtypes

timestamp                 str
age                       str
industry                  str
job_title                 str
job_title_context         str
annual_salary             str
extra_compensation    float64
currency                  str
other_currency            str
income_context            str
country                   str
state                     str
city                      str
overall_exp               str
field_exp                 str
education                 str
gender                    str
race                      str
dtype: object

In [25]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["annual_salary"] = (
    df["annual_salary"].str.strip()
    .str.replace(",", "")
)
df["annual_salary"] = df["annual_salary"].astype("float64")

The column timestamp has been converted from a string to datetime.

The column annual_salary has been converted from string to float.

### <u>Standardizing the values in the columns<u>

### <u>Handling missing values.<u>

In [6]:
df.isna().sum()

timestamp                 0
age                       0
industry                 86
job_title                 5
job_title_context     20930
annual_salary             0
extra_compensation     7374
currency                  0
other_currency        27999
income_context        25170
country                   2
state                  5076
city                     85
overall_exp               0
field_exp                 0
education               241
gender                  186
race                    197
dtype: int64

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

The colums industry, job_title, country, city, education and race have missing values.

Since there is no way of finding appropriate values to fill into these empty slots, it is best to drop the rows that contain missing values in the industry column.

Since the number of rows with missing values in these columns are less than 5%, dropping those should not have any significant impact on any future analysis done.

In [7]:
df = df.dropna(subset= ["industry", "job_title", "city", "country", "education", "race"])

df[["industry", "job_title", "city", "country", "education", "race"]].isna().sum()

industry     0
job_title    0
city         0
country      0
education    0
race         0
dtype: int64

All the rows with missing values in the those columns have been dropped.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Since the job_title_context columns only contains additional context of the job title, it can be concluded that the rows with missing values in this column did not require any aditional context.

These missing values can then be filled with "No additional context"

In [8]:
df["job_title_context"] = df["job_title_context"].fillna("No additional context")

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

The column other_currency has 27,479.

This column should only filled in if the value in the currency column is "Other".

In [9]:
df["currency"].unique()

<StringArray>
[    'USD',     'GBP',     'CAD',     'EUR', 'AUD/NZD',   'Other',     'CHF',
     'ZAR',     'SEK',     'HKD',     'JPY']
Length: 11, dtype: str

All the rows who have the value "Other" in the column currency and have a missing value in the column other_currency should be dropped.

In [10]:
df = df[~((df["currency"] == "Other") & (df["other_currency"].isna()))]

All the rows where the value in the the value in the column currency is not "Other" and the value in the column other_currency is missing will have that missing value filled with "Not Applicable".

In [11]:
df["other_currency"] = np.where(
    (df["currency"] != "Other") & (df["other_currency"].isna()),
    "Not Applicable",
    df["other_currency"]
)

df["other_currency"].isna().sum()

np.int64(0)

All the missing values in the column other_currency have now been cleared.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Since the column income_context only gets filled with a value if there is a need to add more context on the income, the missing values can be filled with "No additional context".

In [12]:
df['income_context'] = df['income_context'].fillna("No additional context")
df["income_context"].isna().sum()

np.int64(0)

All missing values in the income_context column have been filled with "No additional context"

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

The column extra_compensation only needs to get filled in if there is any extra monetary compensation.

Since this is the case, the rows with missing values in this column can be filled in with 0 and we assume that there was no extra monetary compensation.

In [13]:
df['extra_compensation'] = df['extra_compensation'].fillna(0)
df["extra_compensation"].isna().sum()

np.int64(0)

All the rows with missing values in the column extra_compensation have had those missing values filled.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In the column gender, we can assume that the individual answering the survey was not comfortable sharing their gender and fill in the missing values with "Rather not say".

In [14]:
df['gender'] = df['gender'].fillna("Rather not say")
df["gender"].isna().sum()

np.int64(0)

The missing values in the column gender were filled in with the value "Rather not say".

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In the column states, a value is only expected if the contry is in the USA.

If the value in the column country is USA and the value in the column states is missing, it can be filled in with "Not specified".

If the value in the country column is not USA and the value in the states column is missing, it can be filled in with "Not needed".

In [15]:
df['state'] = np.where(
    (df['country'] != 'USA') & (df['state'].isna()),
    'Not needed',
    df['state'].fillna('Not specified')
)

In [16]:
df.isna().sum()

timestamp             0
age                   0
industry              0
job_title             0
job_title_context     0
annual_salary         0
extra_compensation    0
currency              0
other_currency        0
income_context        0
country               0
state                 0
city                  0
overall_exp           0
field_exp             0
education             0
gender                0
race                  0
dtype: int64

There are no more missing values in the dataframe.

### <u>Handling duplicates<u>

### <u>Veryfing the integrity of the data<u>